In [ ]:
import json

from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import get_param_vals_by_analysis, get_prop_better, get_median_scale_strength
from emu_renewal.plotting import plot_scale_exp_map
from emu_renewal.utils import get_analysis_paths

set_matplotlib_formats("svg")

In [ ]:
ANALYSIS = "oxcgrt_floored"
BASELINE = "no_scaling"

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)

strengths = {}
prop_better = {}
for iso3, paths in analysis_paths.items():
    if ANALYSIS not in paths or BASELINE not in paths:
        continue
    strengths[iso3] = get_median_scale_strength(paths[ANALYSIS])
    c_paths = {k: paths[k] for k in (BASELINE, ANALYSIS)}
    disp = get_param_vals_by_analysis("dispersion_proc", c_paths)
    prop_better[iso3] = get_prop_better(disp, ANALYSIS, BASELINE)

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)
world["strength"] = world["ISO_A3"].map(strengths)
supported = world[world["ISO_A3"].isin([i for i, p in prop_better.items() if p > 0.5])]

fig = plot_scale_exp_map(world, "strength", 1.0, supported)
display(fig)

In [ ]:
fig.savefig("figure_6.svg", bbox_inches="tight")